In [ ]:
# Install required packages
# install.packages(c("ggplot2", "ggrepel", "dplyr"))

library(ggplot2)
library(ggrepel)
library(dplyr)

# Read data
df <- read.csv("/path/to/CRISPR/results/CRISPR_results_zlu_lib_FC1_p0.05.csv")

# Rename columns if they contain parentheses or special characters
colnames(df) <- c("sgRNA", "Gene", "avg_control", "avg_treat", 
                  "Fold_change", "log2FC", "p_value", "negLog10P", "significant")

In [ ]:
library(ggplot2)
library(ggrepel)
library(dplyr)

# Threshold settings
fc_cutoff <- 2
p_cutoff <- 0.05
top_n <- 5  # Number of labels per direction

# Classification
df <- df %>%
  mutate(status = case_when(
    log2FC >= fc_cutoff & p_value < p_cutoff ~ "Enriched",
    log2FC <= -fc_cutoff & p_value < p_cutoff ~ "Depleted",
    TRUE ~ "Not significant"
  ))

# Select top genes (both directions)
top_enriched <- df %>%
  filter(status == "Enriched") %>%
  arrange(p_value, desc(log2FC)) %>%
  head(top_n)

top_depleted <- df %>%
  filter(status == "Depleted") %>%
  arrange(p_value, log2FC) %>%
  head(top_n)

# Force-label specific genes (e.g., ANPEP)
highlight_genes <- df %>%
  filter(Gene %in% c("ANPEP"))

# Merge and deduplicate
label_genes <- bind_rows(top_enriched, top_depleted, highlight_genes) %>%
  distinct(Gene, .keep_all = TRUE)

# Color scheme (Cell/Nature style)
color_scheme <- c("Enriched" = "#d73027",  # Red
                  "Depleted" = "#4575b4",  # Blue
                  "Not significant" = "grey70")

# Plot
ggplot(df, aes(x = log2FC, y = -log10(p_value))) +
  geom_point(aes(color = status), size = 4.5, alpha = 0.9) +
  scale_color_manual(values = color_scheme) +
  geom_vline(xintercept = c(-fc_cutoff, fc_cutoff),
             linetype = "dashed", color = "black", linewidth = 0.4) +
  geom_hline(yintercept = -log10(p_cutoff),
             linetype = "dashed", color = "black", linewidth = 0.4) +
  geom_label_repel(
    data = label_genes,
    aes(label = Gene),
    size = 3.5,
    color = "black",
    fill = "white",
    label.size = 0.2,
    box.padding = 0.25,
    point.padding = 0.2,
    segment.color = "black",
    segment.size = 0.3,
    max.overlaps = 100,
    fontface = "bold" 
  ) +
  labs(
    x = expression(Log[2]~"fold change"),
    y = expression(-Log[10]~"p-value"),
    color = "Screen outcome"
  ) +
  theme_bw(base_size = 13) +
theme(
    text = element_text(face = "bold"), 
    panel.border = element_rect(colour = "black", fill = NA, linewidth = 1.2),
    axis.text = element_text(color = "black", face = "bold", size = 12), 
    axis.title = element_text(color = "black", face = "bold", size = 14), 
    axis.ticks.length = unit(0.25, "cm"),
    legend.position = "right",
    legend.title = element_text(size = 12, face = "bold"),
    legend.text = element_text(size = 11, face = "bold"),
    panel.grid.major = element_blank(),
    panel.grid.minor = element_blank()
  )

In [ ]:
ggsave(
  filename = "/path/to/CRISPR/results/CRISPR_volcano.pdf",   # Output file name
  plot = last_plot(),                # Or specify your ggplot object directly, e.g., plot = volcano_plot
  device = cairo_pdf,                # Ensure editable fonts (embed TrueType)
  width = 6, height = 5, units = "in",  # Dimensions (inches)
  dpi = 300                          # Resolution
)

In [ ]:
# Volcano plot with WDR13 and LPP highlighted (no background box)
library(ggplot2)
library(ggrepel)
library(dplyr)

# Threshold settings
fc_cutoff <- 2
p_cutoff <- 0.05
top_n <- 5  # Number of labels per direction

# Classification
df <- df %>%
  mutate(status = case_when(
    log2FC >= fc_cutoff & p_value < p_cutoff ~ "Enriched",
    log2FC <= -fc_cutoff & p_value < p_cutoff ~ "Depleted",
    TRUE ~ "Not significant"
  ))

# Select top genes (both directions)
top_enriched <- df %>%
  filter(status == "Enriched") %>%
  arrange(p_value, desc(log2FC)) %>%
  head(top_n)

top_depleted <- df %>%
  filter(status == "Depleted") %>%
  arrange(p_value, log2FC) %>%
  head(top_n)

# Force-label specific genes (ANPEP, WDR13, LPP)
highlight_genes <- df %>%
  filter(Gene %in% c("ANPEP", "WDR13", "LPP"))

# Check if these genes exist in the data
cat("Genes to highlight:\n")
print(highlight_genes %>% select(Gene, log2FC, p_value, status))

# Merge and deduplicate
label_genes <- bind_rows(top_enriched, top_depleted, highlight_genes) %>%
  distinct(Gene, .keep_all = TRUE)

# Color scheme (Cell/Nature style)
color_scheme <- c("Enriched" = "#d73027",  # Red
                  "Depleted" = "#4575b4",  # Blue
                  "Not significant" = "grey70")

# Plot
volcano_plot <- ggplot(df, aes(x = log2FC, y = -log10(p_value))) +
  geom_point(aes(color = status), size = 2.5, alpha = 0.9) +
  scale_color_manual(values = color_scheme) +
  geom_vline(xintercept = c(-fc_cutoff, fc_cutoff),
             linetype = "dashed", color = "black", linewidth = 0.4) +
  geom_hline(yintercept = -log10(p_cutoff),
             linetype = "dashed", color = "black", linewidth = 0.4) +
  geom_text_repel(
    data = label_genes,
    aes(label = Gene),
    size = 4,
    color = "black",
    box.padding = 0.5,
    point.padding = 0.3,
    segment.color = "black",
    segment.size = 0.4,
    segment.alpha = 0.8,
    min.segment.length = 0,
    max.overlaps = 100,
    fontface = "bold.italic"
  ) +
  labs(
    x = expression(Log[2]~"fold change"),
    y = expression(-Log[10]~"p-value"),
    color = "Screen outcome"
  ) +
  theme_bw(base_size = 13) +
  theme(
    text = element_text(face = "bold"), 
    panel.border = element_rect(colour = "black", fill = NA, linewidth = 1.2),
    axis.text = element_text(color = "black", face = "bold", size = 12), 
    axis.title = element_text(color = "black", face = "bold", size = 14), 
    axis.ticks.length = unit(0.25, "cm"),
    legend.position = "right",
    legend.title = element_text(size = 12, face = "bold"),
    legend.text = element_text(size = 11, face = "bold"),
    panel.grid.major = element_blank(),
    panel.grid.minor = element_blank()
  )

# Display plot
print(volcano_plot)

# Save plot
ggsave(
  filename = "/path/to/CRISPR/results/CRISPR_volcano_with_WDR13_LPP.pdf",
  plot = volcano_plot,
  device = cairo_pdf,
  width = 6, height = 5, units = "in",
  dpi = 300
)

cat("\nVolcano plot saved to: CRISPR_volcano_with_WDR13_LPP.pdf\n")

In [ ]:
# Volcano plot using complete CRISPR library (all 119 genes including LPP)
library(ggplot2)
library(ggrepel)
library(dplyr)

# Read complete data
df_complete <- read.csv("/path/to/CRISPR/results/crispr_rnaseq_merged.csv")

# Rename columns for convenience
colnames(df_complete) <- c("gene_id", "Gene", "log2FC", "category", "rna_log2fc", "rna_padj", "rna_deg_status")

# Create pseudo p-values based on category (for y-axis display)
# Add random jitter so genes in the same category are dispersed on the y-axis
set.seed(42)  # Set random seed for reproducibility

df_complete <- df_complete %>%
  mutate(
    # Base p-value
    base_pvalue = case_when(
      category == "Highly Enriched" ~ 0.001,
      category == "Moderately Enriched" ~ 0.01,
      category == "Slightly Enriched" ~ 0.05,
      category == "Highly Depleted" ~ 0.001,
      category == "Moderately Depleted" ~ 0.01,
      category == "Slightly Depleted" ~ 0.05,
      category == "Depleted" ~ 0.01,
      TRUE ~ 0.1
    ),
    # Add random jitter (within the same order of magnitude)
    jitter_factor = runif(n(), 0.5, 2.0),
    pseudo_pvalue = base_pvalue * jitter_factor
  )

# Threshold settings
fc_cutoff <- 2
p_cutoff <- 0.05
top_n <- 5

# Classification
df_complete <- df_complete %>%
  mutate(status = case_when(
    grepl("Enriched", category, ignore.case = TRUE) & log2FC >= fc_cutoff ~ "Enriched",
    grepl("Depleted", category, ignore.case = TRUE) & log2FC <= -fc_cutoff ~ "Depleted",
    TRUE ~ "Not significant"
  ))

# Select top genes
top_enriched <- df_complete %>%
  filter(status == "Enriched") %>%
  arrange(pseudo_pvalue, desc(log2FC)) %>%
  head(top_n)

top_depleted <- df_complete %>%
  filter(status == "Depleted") %>%
  arrange(pseudo_pvalue, log2FC) %>%
  head(top_n)

# Label specific genes
highlight_genes <- df_complete %>%
  filter(Gene %in% c("ANPEP", "WDR13", "LPP"))

cat("Genes to highlight:\n")
print(highlight_genes %>% select(Gene, log2FC, category, status))

# Merge
label_genes <- bind_rows(top_enriched, top_depleted, highlight_genes) %>%
  distinct(Gene, .keep_all = TRUE)

# Color scheme
color_scheme <- c("Enriched" = "#d73027", "Depleted" = "#4575b4", "Not significant" = "grey70")

# Plot
volcano_complete <- ggplot(df_complete, aes(x = log2FC, y = -log10(pseudo_pvalue))) +
  geom_point(aes(color = status), size = 2.5, alpha = 0.9) +
  scale_color_manual(values = color_scheme) +
  geom_vline(xintercept = c(-fc_cutoff, fc_cutoff), linetype = "dashed", color = "black", linewidth = 0.4) +
  geom_hline(yintercept = -log10(p_cutoff), linetype = "dashed", color = "black", linewidth = 0.4) +
  geom_text_repel(
    data = label_genes,
    aes(label = Gene),
    size = 4,
    color = "black",
    box.padding = 0.5,
    point.padding = 0.3,
    segment.color = "black",
    segment.size = 0.4,
    segment.alpha = 0.8,
    min.segment.length = 0,
    max.overlaps = 100,
    fontface = "bold.italic"
  ) +
  labs(
    x = expression(Log[2]~"fold change"),
    y = expression(-Log[10]~"(significance)"),
    color = "Screen outcome",
    title = "CRISPR Screen - Complete Library (119 genes)"
  ) +
  theme_bw(base_size = 13) +
  theme(
    text = element_text(face = "bold"),
    panel.border = element_rect(colour = "black", fill = NA, linewidth = 1.2),
    axis.text = element_text(color = "black", face = "bold", size = 12),
    axis.title = element_text(color = "black", face = "bold", size = 14),
    axis.ticks.length = unit(0.25, "cm"),
    legend.position = "right",
    legend.title = element_text(size = 12, face = "bold"),
    legend.text = element_text(size = 11, face = "bold"),
    panel.grid.major = element_blank(),
    panel.grid.minor = element_blank(),
    plot.title = element_text(hjust = 0.5, size = 15)
  )

print(volcano_complete)

# Save
ggsave(
  filename = "/path/to/CRISPR/results/CRISPR_volcano_complete_library.pdf",
  plot = volcano_complete,
  device = cairo_pdf,
  width = 6, height = 5, units = "in",
  dpi = 300
)

cat("\nComplete library volcano plot saved!\n")